# How the comparison area changes apparent performance

The precision of a constant score equals the fraction of mapped water in the evaluation population. Compare this reference level with the average precision of the same frozen terrain model. The wider Lokoja check includes locations where prior water history is unknown. A larger geographic contrast can make ranking easier.

In [1]:
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt

root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'pyproject.toml').exists())
out = root/'reports/nigeria_gfd'
def read(name):
    return json.loads((out/name).read_text())

historical = read('evaluation.json')
restricted = read('lokoja_2022.json')
expanded = read('lokoja_2022_expanded.json')
baseline = [historical['partitions']['test']['weighted_reference_fraction']] + [
    row['mapped_water']/row['eligible'] for row in [restricted, expanded]]
model = [historical['test']['calibrated_terrain_gbm']['average_precision']] + [
    row['calibrated']['average_precision'] for row in [restricted, expanded]]
y = np.arange(3)
fig, ax = plt.subplots(figsize=(8, 3.5))
for values, offset, color, label in [(baseline, 0.17, '#87949c', 'Constant score'), (model, -0.17, '#1a6474', 'Terrain model')]:
    bars = ax.barh(y+offset, values, height=0.30, color=color, label=label)
    ax.bar_label(bars, fmt='%.3f', padding=4)
ax.set_yticks(y, ['2018 event', 'Lokoja · prior class known', 'Lokoja · unknown included'])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel('Average precision')
ax.set_title('Terrain ranking against satellite-mapped water')
ax.legend(loc='lower right', frameon=False)
fig.tight_layout()
fig.savefig(out/'terrain_reference_precision.png', dpi=160, bbox_inches='tight')
plt.close(fig)